In [ ]:
!pip install requests pandas openpyxl tqdm -q

In [ ]:
import requests
import pandas as pd
import time
import json
from tqdm import tqdm
from statistics import median

In [ ]:
OPENROUTER_API_KEY = "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"

In [ ]:
# Models
Models = [
    "meta-llama/llama-3-8b-instruct",
    "mistralai/mistral-7b-instruct",
    "google/gemma-3-12b-it"
]

In [ ]:
# Prompts
prompts = [
    {
        "id":"p1",
        "category":"Factual QA",
        "prompt":"Who invented Java programming language?"
    },
    {
        "id": "P2",
        "category": "Reasoning",
        "prompt": "If a train travels 60 km in 1 hour, how much distance will it cover in 5 hours?"
    },
    {
         "id": "P4",
        "category": "Instruction Following",
        "prompt": "Explain recursion in exactly 2 lines."
    },
    {
         "id": "P5",
        "category": "Coding",
        "prompt": "Write Java code to reverse an array."
    }
]

In [ ]:
def querymodel(model_name,user_prompt):
  url="https://openrouter.ai/api/v1/chat/completions"

  headers={
      "Authorization": f"Bearer {OPENROUTER_API_KEY}",
      "Content-Type":"application/json"
  }

  payload = {
      "model":model_name,
      "messages":[
                 {
                "role": "system",
                "content": "You are a helpful AI assistant."
            },
            {
                "role": "user",
                "content": user_prompt
            }
      ],

      "temperature":0.2,
      "max_tokens":300
  }

  start_time = time.time()

  response = requests.post(url,headers=headers,json=payload)

  end_time = time.time()

  latency_ms = round((end_time - start_time) * 1000, 2)

  if response.status_code == 200:

        result = response.json()

        try:
            content = result["choices"][0]["message"]["content"]

            usage = result.get("usage", {})

            input_tokens = usage.get("prompt_tokens", 0)
            output_tokens = usage.get("completion_tokens", 0)

            return {
                "response": content,
                "latency": latency_ms,
                "input_tokens": input_tokens,
                "output_tokens": output_tokens,
                "success": True
            }

        except Exception as e:
            return {
                "response": f"Parsing Error: {str(e)}",
                "latency": latency_ms,
                "input_tokens": 0,
                "output_tokens": 0,
                "success": False
            }

  else:
        return {
            "response": f"API Error: {response.text}",
            "latency": latency_ms,
            "input_tokens": 0,
            "output_tokens": 0,
            "success": False
        }

In [ ]:
results = []

for prompt_data in tqdm(prompts):

    prompt_id = prompt_data["id"]
    category = prompt_data["category"]
    prompt_text = prompt_data["prompt"]

    for model in Models:

        print(f"\nRunning {model} on {prompt_id}")

        output = querymodel(model, prompt_text)

        correctness = 0
        instruction_following = 0
        completeness = 0
        factuality = 0
        groundedness = 0
        relevance = 0
        hallucination = 0
        safety = 0

        notes = ""
        results.append({
            "Prompt ID": prompt_id,
            "Prompt Category": category,
            "Prompt": prompt_text,
            "Model Name": model,
            "Model Response": output["response"],
            "Answer Correctness": correctness,
            "Instruction Following": instruction_following,
            "Completeness": completeness,
            "Factuality": factuality,
            "Faithfulness/Groundedness": groundedness,
            "Answer Relevance": relevance,
            "Hallucination Score": hallucination,
            "Safety/Refusal Accuracy": safety,
            "Latency ms": output["latency"],
            "Input Tokens": output["input_tokens"],
            "Output Tokens": output["output_tokens"],
            "Successful Answer?": output["success"],
            "Notes": notes
        })

  0%|          | 0/4 [00:00<?, ?it/s]


Running meta-llama/llama-3-8b-instruct on p1

Running mistralai/mistral-7b-instruct on p1

Running google/gemma-3-12b-it on p1


 25%|██▌       | 1/4 [00:05<00:17,  5.93s/it]


Running meta-llama/llama-3-8b-instruct on P2

Running mistralai/mistral-7b-instruct on P2

Running google/gemma-3-12b-it on P2


 50%|█████     | 2/4 [00:09<00:08,  4.50s/it]


Running meta-llama/llama-3-8b-instruct on P4

Running mistralai/mistral-7b-instruct on P4

Running google/gemma-3-12b-it on P4


 75%|███████▌  | 3/4 [00:11<00:03,  3.35s/it]


Running meta-llama/llama-3-8b-instruct on P5

Running mistralai/mistral-7b-instruct on P5

Running google/gemma-3-12b-it on P5


100%|██████████| 4/4 [00:23<00:00,  5.81s/it]


In [ ]:
df = pd.DataFrame(results)

In [ ]:
csv_file = "model_evaluation_results.csv"
excel_file = "model_evaluation_results.xlsx"

df.to_csv(csv_file, index=False)
df.to_excel(excel_file, index=False)

print("\n")
print("EVALUATION COMPLETED")
print(" ")

print(f"\nCSV Saved: {csv_file}")
print(f"Excel Saved: {excel_file}")



EVALUATION COMPLETED
 

CSV Saved: model_evaluation_results.csv
Excel Saved: model_evaluation_results.xlsx


In [ ]:
df.head()

,Prompt ID,Prompt Category,Prompt,Model Name,Model Response,Answer Correctness,Instruction Following,Completeness,Factuality,Faithfulness/Groundedness,Answer Relevance,Hallucination Score,Safety/Refusal Accuracy,Latency ms,Input Tokens,Output Tokens,Successful Answer?,Notes
0,p1,Factual QA,Who invented Java programming language?,meta-llama/llama-3-8b-instruct,"Java was invented by James Gosling, Mike Sheri...",0,0,0,0,0,0,0,0,1168.84,28,174,True,
1,p1,Factual QA,Who invented Java programming language?,mistralai/mistral-7b-instruct,"API Error: {""error"":{""message"":""No endpoints f...",0,0,0,0,0,0,0,0,30.16,0,0,False,
2,p1,Factual QA,Who invented Java programming language?,google/gemma-3-12b-it,The Java programming language was invented by ...,0,0,0,0,0,0,0,0,4730.90,23,137,True,
3,P2,Reasoning,"If a train travels 60 km in 1 hour, how much d...",meta-llama/llama-3-8b-instruct,"If the train travels 60 km in 1 hour, we can f...",0,0,0,0,0,0,0,0,780.38,45,93,True,
4,P2,Reasoning,"If a train travels 60 km in 1 hour, how much d...",mistralai/mistral-7b-instruct,"API Error: {""error"":{""message"":""No endpoints f...",0,0,0,0,0,0,0,0,32.74,0,0,False,


In [ ]:
summary = []

for model in Models:
    model_df = df[df["Model Name"]==model]

    latencies = list(model_df["Latency ms"])

    if len(latencies) > 0:

        p50 = median(latencies)

        p95 = sorted(latencies)[int(len(latencies) * 0.95) - 1]

        summary.append({
            "Model": model,
            "p50 Latency": p50,
            "p95 Latency": p95
        })

summary_df = pd.DataFrame(summary)

print("\n")
print("LATENCY SUMMARY")
print("")

display(summary_df)




LATENCY SUMMARY



,Model,p50 Latency,p95 Latency
0,meta-llama/llama-3-8b-instruct,974.61,1168.84
1,mistralai/mistral-7b-instruct,31.45,32.74
2,google/gemma-3-12b-it,3706.44,4730.90


In [ ]:
from google.colab import files

files.download(csv_file)
files.download(excel_file)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>